# 📘 Notebook 1 — RAG Without LangChain
### No Framework · No UI · Every Step Visible

**What you will see:** The complete RAG pipeline written in pure Python.
No LangChain. No abstractions. Every function is yours.

**Why start here:** You cannot understand what LangChain does for you until
you have built the same thing yourself.

**Stack:** Groq (LLaMA 3.3 70B) · HuggingFace sentence-transformers · FAISS · Pure Python

---
### The Pipeline You Are Building:
```
Documents → Chunk → Embed → FAISS Index
Question  → Embed → Retrieve Top-k Chunks → Build Prompt → Groq → Answer
```


In [ ]:
# ── CELL 1: Install ──────────────────────────────────────────────────────────
!pip install sentence-transformers faiss-cpu openai python-dotenv numpy -q
print("✅ Done")


In [ ]:
# ── CELL 2: Set API Key ──────────────────────────────────────────────────────
import os

# Paste your Groq API key here
os.environ["GROQ_API_KEY"] = "enter-your-api-key-here"

print("✅ API key set")
print("   Get a free key at: https://console.groq.com/keys")


In [ ]:
# ── CELL 3: Imports ──────────────────────────────────────────────────────────
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# LLM via Groq — uses OpenAI-compatible client
# base_url points to Groq instead of OpenAI
LLM_CLIENT = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
LLM_MODEL = "openai/gpt-oss-120b"

print("✅ Imports done")
print(f"   LLM: {LLM_MODEL} via Groq (free tier available)")


In [ ]:
# ── CELL 4: Load Embedding Model ─────────────────────────────────────────────
# all-MiniLM-L6-v2 is a free, local embedding model from HuggingFace
# Downloads once (~80MB), cached after that, NO API key required
#
# WANT PAID EMBEDDINGS INSTEAD?
# from openai import OpenAI as OAI
# oa = OAI(api_key="your-openai-key")
# def get_embedding(text):
#     r = oa.embeddings.create(input=text, model="text-embedding-3-small")
#     v = np.array(r.data[0].embedding, dtype=np.float32)
#     return v / np.linalg.norm(v)
# DIMENSION = 1536  ← update this too

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
DIMENSION = 384   # number of dimensions this model outputs

print("⏳ Loading embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"✅ Embedding model ready: {EMBED_MODEL_NAME}")
print(f"   Output dimensions: {DIMENSION}")
print(f"   This model runs entirely on your machine — no API needed")


In [ ]:
# ── CELL 5: Sample Documents ─────────────────────────────────────────────────
# These are the documents our RAG system will answer questions about.
# In a real system these would come from PDFs, databases, websites, etc.
# We use text here to keep things clear and focused on the pipeline.

RAW_DOCUMENTS = [
    {
        "title": "What is RAG",
        "text": """RAG stands for Retrieval-Augmented Generation.
It is a technique where an AI model retrieves relevant documents
before generating an answer. RAG helps the model answer questions
about data it was not trained on. This reduces hallucinations and
keeps answers grounded in real content from your own documents.
The two phases are: indexing (done once) and retrieval+generation (done on every query)."""
    },
    {
        "title": "LangChain Framework",
        "text": """LangChain is a framework for developing applications powered by language models.
It provides tools to connect LLMs with external data sources and APIs.
In LangChain v1, legacy chains were removed from the core package.
The modern approach uses LCEL (LangChain Expression Language) to build pipelines.
LCEL uses the pipe operator | to chain components together.
LangChain provides pre-built loaders, splitters, embeddings, and vector store wrappers."""
    },
    {
        "title": "FAISS Vector Database",
        "text": """FAISS is a library developed by Facebook AI for efficient similarity search.
It is commonly used as a vector store in RAG pipelines.
FAISS stores embedding vectors and retrieves the most similar ones for a given query.
IndexFlatIP performs exact inner product (cosine) similarity search.
For large datasets, IndexHNSWFlat provides approximate but much faster search.
FAISS runs entirely in memory — there is no server or database to set up."""
    },
    {
        "title": "Embeddings Explained",
        "text": """An embedding is a list of numbers that represents the meaning of text.
Sentences with similar meanings produce similar vectors.
The all-MiniLM-L6-v2 model produces 384-dimensional vectors for free.
OpenAI's text-embedding-3-small produces 1536-dimensional vectors for a small cost.
Cosine similarity measures how close two embeddings are: 1.0 = identical, 0.0 = unrelated.
The embedding model used for indexing and querying must always be the same model."""
    },
    {
        "title": "Groq API",
        "text": """Groq provides fast inference for open-source models like LLaMA and Gemma.
The Groq API is compatible with the OpenAI client — just change the base_url.
LLaMA 3.3 70B is a powerful open-source model available free on Groq's tier.
Response times on Groq are extremely fast due to their custom LPU hardware.
Get a free API key at console.groq.com/keys — no credit card required."""
    },
]

print(f"✅ {len(RAW_DOCUMENTS)} documents loaded")
for doc in RAW_DOCUMENTS:
    print(f"   • {doc['title']} ({len(doc['text'])} chars)")


In [ ]:
# ── CELL 6: Chunk Documents ──────────────────────────────────────────────────
# WHY CHUNK: LLMs have token limits. Also, smaller chunks = more precise
# retrieval (find exactly the right paragraph, not a whole chapter).
# WHY OVERLAP: prevents context being lost when a sentence falls at a boundary.

CHUNK_SIZE = 300     # characters per chunk
CHUNK_OVERLAP = 50  # characters shared between adjacent chunks

def chunk_text(text: str, source_title: str) -> list:
    """Split text into overlapping chunks with metadata."""
    chunks = []
    start = 0
    text = text.strip()

    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk = text[start:end].strip()

        if len(chunk) > 30:   # skip very short fragments
            chunks.append({
                "text":   chunk,
                "source": source_title,
                "chunk_id": len(chunks),
            })

        if end >= len(text):
            break
        start += CHUNK_SIZE - CHUNK_OVERLAP  # move forward but overlap

    return chunks


# Chunk all documents
all_chunks = []
for doc in RAW_DOCUMENTS:
    doc_chunks = chunk_text(doc["text"], doc["title"])
    all_chunks.extend(doc_chunks)

print(f"✅ Chunked into {len(all_chunks)} pieces")
print(f"   chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print(f"\nExample chunk:")
print(f'  Source: {all_chunks[0]["source"]}')
print(f'  Text:   {all_chunks[0]["text"][:120]}...')


In [ ]:
# ── CELL 7: Embed All Chunks + Build FAISS Index ─────────────────────────────
# This is the INDEXING PHASE — done once.
# Each chunk of text → a 384-dimensional vector → stored in FAISS.

def get_embedding(text: str) -> np.ndarray:
    """Convert text to a normalised float32 vector."""
    vec = embed_model.encode(text, normalize_embeddings=True)
    return vec.astype(np.float32)


def build_index(chunks: list):
    """Embed all chunks and store in a FAISS index."""
    print(f"  Embedding {len(chunks)} chunks...")

    # Encode all texts in one batch (faster than one by one)
    texts = [c["text"] for c in chunks]
    vectors = embed_model.encode(texts, normalize_embeddings=True,
                                  show_progress_bar=True)
    vectors = np.array(vectors, dtype=np.float32)

    # IndexFlatIP = exact cosine similarity (since vectors are normalized)
    index = faiss.IndexFlatIP(DIMENSION)
    index.add(vectors)

    return index


print("Building FAISS index...")
faiss_index = build_index(all_chunks)

print(f"\n✅ FAISS index built")
print(f"   Vectors stored: {faiss_index.ntotal}")
print(f"   Dimensions:     {DIMENSION}")
print(f"   Index type:     IndexFlatIP (exact cosine similarity)")


In [ ]:
# ── CELL 8: Retrieval Function ────────────────────────────────────────────────
# RETRIEVAL PHASE — runs on every user question.
# 1. Embed the question
# 2. Find the most similar vectors in FAISS
# 3. Return the corresponding text chunks

TOP_K = 3   # number of chunks to retrieve per question

def retrieve(query: str, top_k: int = TOP_K) -> list:
    """
    Find the top-k most relevant chunks for a query.

    Steps:
    1. Embed the query with the SAME model used during indexing
    2. Search FAISS for nearest vectors
    3. Return matched chunks with scores
    """
    query_vec = get_embedding(query).reshape(1, -1)  # FAISS needs 2D array

    scores, indices = faiss_index.search(query_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:  # FAISS returns -1 when fewer results than top_k
            continue
        results.append({
            **all_chunks[idx],
            "score": float(score)
        })

    return results


# Test retrieval
print("Testing retrieval...")
test_results = retrieve("What is RAG and how does it work?")
print(f"\nQuery: 'What is RAG and how does it work?'")
print(f"Retrieved {len(test_results)} chunks:\n")
for r in test_results:
    print(f"  Score: {r['score']:.3f} | Source: {r['source']}")
    print(f"  Text:  {r['text'][:80]}...")
    print()


In [ ]:
# ── CELL 9: Generate Answer (RAG) ────────────────────────────────────────────
# GENERATION PHASE — takes retrieved chunks + question → LLM → answer.
# The prompt explicitly tells the LLM to ONLY use the provided context.
# This is the key to preventing hallucinations.

def generate_answer(question: str, retrieved_chunks: list) -> str:
    """
    Build a RAG prompt and call Groq LLM.

    The prompt has 3 parts:
    1. System instruction: answer ONLY from context, admit when unsure
    2. Context: the retrieved document chunks (the 'open book')
    3. Question: the user's question
    """
    if not retrieved_chunks:
        return "⚠️ No relevant documents found. Please try a different question."

    # Format retrieved chunks into a readable context block
    context = "\n\n".join(
        f"[Source: {c['source']} | Relevance: {c['score']:.2f}]\n{c['text']}"
        for c in retrieved_chunks
    )

    prompt = f"""You are a helpful assistant. Answer ONLY using the context provided below.
If the answer is not in the context, say exactly: "I don't have that information."
Do NOT make up any facts. Be concise and clear.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    response = LLM_CLIENT.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,    # low temperature = factual, consistent
        max_tokens=500,
    )

    return response.choices[0].message.content


print("✅ generate_answer() function ready")


In [ ]:
# ── CELL 10: The Complete ask() Function ─────────────────────────────────────
# This ties everything together:
# retrieve() → generate_answer() → print result with sources

def ask(question: str, show_chunks: bool = True):
    """
    Ask any question. The system retrieves relevant chunks and answers from them.

    Parameters:
        question   : the user's question
        show_chunks: if True, show which document chunks were used
    """
    print(f"\n{'='*60}")
    print(f"QUESTION: {question}")
    print(f"{'='*60}")

    # Step 1: Retrieve
    retrieved = retrieve(question)

    # Step 2: Generate
    answer = generate_answer(question, retrieved)

    # Step 3: Display
    print(f"\nANSWER:\n{answer}")

    if show_chunks:
        print(f"\nSOURCE CHUNKS USED:")
        for i, chunk in enumerate(retrieved, 1):
            print(f"  [{i}] {chunk['source']} (score: {chunk['score']:.3f})")
            print(f"       {chunk['text'][:80]}...")

    print(f"{'='*60}\n")


print("✅ ask() function ready — run the next cells to test")


In [ ]:
# ── CELL 11: Test Questions ───────────────────────────────────────────────────

# Question 1: Directly in our documents
ask("What is RAG and why is it useful?")


In [ ]:
# Question 2: Requires understanding LangChain content
ask("What changed in LangChain v1 regarding chains?")


In [ ]:
# Question 3: Tests FAISS knowledge
ask("What is FAISS and where does it come from?")


In [ ]:
# Question 4: Outside our documents — should admit it doesn't know
ask("Who is the CEO of Groq?")


In [ ]:
# ── CELL 12: Understand What Just Happened ───────────────────────────────────

print("WHAT JUST HAPPENED — Step by Step:")
print()
print("INDEXING (ran in cells 6-7):")
print("  1. Loaded 5 text documents")
print("  2. Split them into", len(all_chunks), "chunks of ~300 chars each")
print("  3. Embedded each chunk → 384-dim vector using all-MiniLM-L6-v2")
print("  4. Stored all", faiss_index.ntotal, "vectors in FAISS")
print()
print("RETRIEVAL + GENERATION (ran for each question):")
print("  5. Embedded the question → 384-dim vector")
print("  6. FAISS found the", TOP_K, "most similar vectors (cosine similarity)")
print("  7. Retrieved the matching text chunks + their sources")
print("  8. Built a prompt: instruction + context chunks + question")
print("  9. Sent to Groq LLaMA 3.3 70B → got a grounded answer")
print()
print("WHAT THIS DOES NOT HAVE:")
print("  ❌ Conversation memory (follow-up questions break)")
print("  ❌ LangChain components (everything is manual)")
print("  ❌ UI (terminal only)")
print()
print("NEXT: Notebook 2 rebuilds this exact pipeline using LangChain.")
